# Clustering Using Spherical K-means

## Set up Environment

### Install Dependencies

In [ ]:
!pip install --upgrade numpy pandas matplotlib wordcloud scikit-learn tqdm

In [2]:
!pip install --upgrade faiss-cpu

### Import Dependencies

In [1]:
import math
import os
import re
import faiss
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from wordcloud import WordCloud
from sklearn.metrics import silhouette_score
from tqdm import tqdm

### Load Data

In [ ]:
df = pd.read_pickle('datasets/tickets.pkl')

X_embeddings = np.vstack(df['EMBEDDING'].values)
embedding_dimension = X_embeddings.shape[1]

print("Vektor embedding berhasil dimuat kembali.")
print(f"Jumlah data: {X_embeddings.shape[0]}")
print(f"Dimensi embedding: {embedding_dimension}")

Vektor embedding berhasil dimuat kembali.
Jumlah data: 1639
Dimensi embedding: 256


0    crawlback comment. bantuan nya crawlback comme...
1    it, menemukan link megapolitan.kompas.com tida...
2    data postingan instagram tidak masuk. it, dice...
3       info siputri tidak bisa akses mas, minta yaaa.
4    dashboard loading. it, keluhan pihak heinz das...
Name: WORDCLOUD TICKET, dtype: str

## Experimenting

### Normalize Data

In [5]:
X_embeddings = np.ascontiguousarray(X_embeddings, dtype=np.float32)
faiss.normalize_L2(X_embeddings)

print("Vektor embedding berhasil dinormalisasi.")

Vektor embedding berhasil dinormalisasi.


### Initialize Spherical K-means

In [6]:
def run_skm(dimension, ncluster, X, niter=20):
    skm = faiss.Kmeans(
        d=dimension, 
        k=ncluster, 
        niter=niter, 
        verbose=True, 
        spherical=True
        )
    skm.train(X)

    centroids = skm.centroids
    _, labels = skm.index.search(X, 1)

    return centroids, labels.flatten()

In [11]:
def generate_wordclouds(texts_list, titles, save_path, main_title=""):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    n_clouds = len(texts_list)
    if n_clouds == 0:
        return
        
    # Menentukan grid (maksimal 3 kolom agar gambar tidak terlalu memanjang)
    cols = min(3, n_clouds)
    rows = math.ceil(n_clouds / cols)
    
    # Membuat figure matplotlib
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4.5))
    if main_title:
        fig.suptitle(main_title, fontsize=20, y=1.02)
        
    # Flatten axes agar mudah diiterasi (untuk kasus 1D atau 2D array)
    if n_clouds == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
        
    for i in range(len(axes)):
        if i < n_clouds:
            combined_text = texts_list[i]
            if combined_text.strip():
                # Generate wordcloud
                wc = WordCloud(
                    width=800, height=400,
                    background_color='white',
                    max_words=100,
                    collocations=False
                ).generate(combined_text)
                
                # Tampilkan di subplot
                axes[i].imshow(wc, interpolation='bilinear')
            else:
                # Jika tidak ada teks untuk klaster ini
                axes[i].text(0.5, 0.5, 'Tidak ada teks', ha='center', va='center', fontsize=14)
                
            axes[i].set_title(titles[i], fontsize=16, pad=10)
            axes[i].axis('off')
        else:
            # Sembunyikan subplot yang kosong (jika jumlah n_clouds tidak genap dengan grid)
            axes[i].axis('off')
            
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.close(fig) # Tutup figure untuk menghemat memori

In [14]:
def cosine_similarity_intra_cluster(embeddings, labels, centroids):
    centroids_norm = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
    sims = np.sum(embeddings * centroids_norm[labels], axis=1)
    return float(np.mean(sims))

### Run Experiments Scenario

In [ ]:
min_clusters = 2
max_clusters = 10
k_range = range(min_clusters, max_clusters + 1)

results = []
scenario_count = 0
wordcloud_texts = df['WORDCLOUD TICKET'].astype(str).values

total_iterations = sum(k1 * len(k_range) for k1 in k_range)

with tqdm(total=total_iterations, desc="Evaluasi + WordCloud") as pbar:
    
    for k1 in k_range:
        centroids_l1, labels_l1 = run_skm(embedding_dimension, k1, X_embeddings)
        silhouette_avg_l1 = silhouette_score(X_embeddings, labels_l1, metric='cosine')

        # --- WordCloud Level 1 Gabungan ---
        l1_texts_list = []
        l1_titles = []
        cluster_texts_dict = {} # Menyimpan teks tiap cluster untuk digunakan di L2
        
        for cluster_id in range(k1):
            cluster_indices = np.where(labels_l1 == cluster_id)[0]
            cluster_texts = wordcloud_texts[cluster_indices].tolist()
            cluster_texts_dict[cluster_id] = (cluster_embeddings:=X_embeddings[cluster_indices], cluster_texts)
            
            l1_texts_list.append(" ".join(cluster_texts))
            l1_titles.append(f"Cluster L1 - {cluster_id + 1}")
            
        generate_wordclouds(
            l1_texts_list, 
            l1_titles, 
            f"results/wordclouds/k{k1}/L1_combined.png",
            main_title=f"Level 1 Wordclouds (k={k1})"
        )

        # Lanjut ke Level 2 menggunakan data yang sudah disiapkan
        for cluster_id in range(k1):
            cluster_embeddings, cluster_texts = cluster_texts_dict[cluster_id]

            for k2 in k_range:
                if len(cluster_embeddings) >= k2:
                    centroids_l2, labels_l2 = run_skm(embedding_dimension, k2, cluster_embeddings)
                    silhouette_avg_l2 = silhouette_score(cluster_embeddings, labels_l2, metric='cosine')

                    # --- Ending Conditions Check ---

                    # Objective Condition 1: jumlah data vector input = jumlah objek yang sudah diklastering
                    n_input_l2 = len(cluster_embeddings)
                    n_clustered_l2 = len(labels_l2)
                    objective_condition_1 = "YES" if n_input_l2 == n_clustered_l2 else "NO"

                    # Subjective Concise: k1 dan k2 berada dalam rentang [2, 10]
                    subjective_concise = "YES" if (2 <= k1 <= 10 and 2 <= k2 <= 10) else "NO"

                    # Subjective Robust: silhouette avg dalam [0, 0.5] DAN cosine similarity intra-cluster mendekati 1
                    cosine_sim_intra_l1 = cosine_similarity_intra_cluster(X_embeddings, labels_l1, centroids_l1)
                    cosine_sim_intra_l2 = cosine_similarity_intra_cluster(cluster_embeddings, labels_l2, centroids_l2)
                    COSINE_SIM_THRESHOLD = 0.75  # ambang "mendekati 1", bisa disesuaikan
                    is_silhouette_in_range = (0 <= silhouette_avg_l1 <= 0.5) and (0 <= silhouette_avg_l2 <= 0.5)
                    is_cosine_near_one = (cosine_sim_intra_l1 >= COSINE_SIM_THRESHOLD) and (cosine_sim_intra_l2 >= COSINE_SIM_THRESHOLD)
                    subjective_robust = "YES" if (is_silhouette_in_range and is_cosine_near_one) else "NO"

                    # Subjective Comprehensive: jumlah objek yang berhasil diklastering = jumlah data vector input
                    n_classified_l2 = np.sum(labels_l2 >= 0)  # label valid (faiss tidak menghasilkan -1, tapi dicek untuk formalitas)
                    subjective_comprehensive = "YES" if n_classified_l2 == n_input_l2 else "NO"

                    results.append({
                        'scenario_id': f"{k1-1}.{cluster_id+1}.{k2-1}",
                        'cluster_id': f"{k1}.{cluster_id+1}",
                        'k1': k1,
                        'k2': k2,
                        'silhouette_avg_l1': f"{silhouette_avg_l1:.4f}",
                        'silhouette_avg_l2': f"{silhouette_avg_l2:.4f}",
                        'cosine_sim_intra_l1': f"{cosine_sim_intra_l1:.4f}",
                        'cosine_sim_intra_l2': f"{cosine_sim_intra_l2:.4f}",
                        'n_input_l2': n_input_l2,
                        'n_clustered_l2': n_clustered_l2,
                        'objective_condition_1': objective_condition_1,
                        'subjective_concise': subjective_concise,
                        'subjective_robust': subjective_robust,
                        'subjective_comprehensive': subjective_comprehensive,
                    })
                    scenario_count += 1

                    # --- WordCloud Level 2 Gabungan ---
                    l2_texts_list = []
                    l2_titles = []
                    
                    for sub_id in range(k2):
                        sub_indices = np.where(labels_l2 == sub_id)[0]
                        sub_texts = [cluster_texts[i] for i in sub_indices]
                        
                        l2_texts_list.append(" ".join(sub_texts))
                        l2_titles.append(f"Sub-cluster L2 - {sub_id + 1}")
                        
                    generate_wordclouds(
                        l2_texts_list,
                        l2_titles,
                        f"results/wordclouds/k{k1}/cluster_{cluster_id+1}_k{k2}/L2_combined.png",
                        main_title=f"Level 2: k={k2} (Berasal dari L1 Cluster {cluster_id + 1})"
                    )
                
                pbar.update(1)

print(f"\nTotal scenario yang berhasil dievaluasi: {scenario_count}")
print("Semua gambar WordCloud telah digabung dan disimpan di results/wordclouds/")

Evaluasi + WordCloud:   0%|          | 0/486 [00:00<?, ?it/s]

Evaluasi + WordCloud: 100%|██████████| 486/486 [00:13<00:00, 36.66it/s]


Total scenario yang berhasil dievaluasi: 486
Semua gambar WordCloud telah digabung dan disimpan di results/wordclouds/


In [22]:
# ===== Ending Condition: Subjective Extendible =====
# Terpenuhi jika silhouette_avg_l1 adalah BEST untuk k1 tersebut (lintas semua cluster_id yang ada di k1 itu, 
# karena silhouette_avg_l1 sama untuk semua baris dengan k1 yang sama -> otomatis konsisten)
# DAN silhouette_avg_l2 adalah BEST di antara semua k2 untuk cluster_id (L1) yang sama.

results_df_temp = pd.DataFrame(results)
results_df_temp['silhouette_avg_l1'] = results_df_temp['silhouette_avg_l1'].astype(float)
results_df_temp['silhouette_avg_l2'] = results_df_temp['silhouette_avg_l2'].astype(float)

# Best silhouette L1 untuk setiap k1
best_sil_l1_per_k1 = results_df_temp.groupby('k1')['silhouette_avg_l1'].max()

# Best silhouette L2 untuk setiap cluster_id (kombinasi k1 + cluster L1 tertentu)
best_sil_l2_per_cluster = results_df_temp.groupby('cluster_id')['silhouette_avg_l2'].max()

for r in results:
    k1_val = r['k1']
    cluster_id_val = r['cluster_id']
    sil_l1_val = float(r['silhouette_avg_l1'])
    sil_l2_val = float(r['silhouette_avg_l2'])

    is_best_l1 = (sil_l1_val == best_sil_l1_per_k1[k1_val])
    is_best_l2 = (sil_l2_val == best_sil_l2_per_cluster[cluster_id_val])

    r['subjective_extendible'] = "YES" if (is_best_l1 and is_best_l2) else "NO"

print("Kolom 'subjective_extendible' berhasil ditambahkan ke seluruh hasil skenario.")

Kolom 'subjective_extendible' berhasil ditambahkan ke seluruh hasil skenario.


In [23]:
file_path = 'results/experiments/results.csv'
os.makedirs(os.path.dirname(file_path), exist_ok=True)

if results:
    fieldnames = list(results[0].keys())
    
    with open(file_path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        
        writer.writeheader()
        writer.writerows(results)
    
    print(f"Results saved to {file_path}")

Results saved to results/experiments/results.csv
